# 27 -- Refit the calibrated blend for the new per-subject-centered CNN (2026-09-13)

Following `notebooks/26_striatum_coverage_retrain.ipynb`'s decisive result
and the 6th Opus review's recommendation (project memory): ship a single
per-subject-centered CNN variant (`model.PRODUCTION_VARIANT_PREFIXES =
["coveragefix_persubject"]`, 25 checkpoints) in place of the old 6-variant/
150-checkpoint ensemble entirely -- re-adding the old variants HURTS the
blend (Opus measured 0.3375 with them vs. 0.3076 without).

This notebook (1) refits `notebooks/22_calibration_refit_rowwise_cv.ipynb`'s
exact method (logit-space pooling + `LogisticRegression(fit_intercept=True)`
on `[logit(cnn_p), logit(baseline_p)]`, row-wise 5-fold CV) on the new
composition, gates it against the shipped 6-variant recipe via
`evaluate.paired_gate`, and produces the full-data-fit constants
(`a, b, c, a1, c1`) to hardcode into `submission_src/main.py`; (2) runs the
one check that de-risks shipping `data.load_volume(uid, center_mm="auto")`
at inference: recomputing `features.striatum_center_mm` via the exact
production code path for a sample of training uids and asserting an EXACT
match against `notebooks/25`'s op02audit measurements (the 25 shipped
checkpoints were trained on those measurements -- if inference's own
computation of the same quantity diverges even slightly, do not ship).

In [1]:
# [RUN ME] -- loads real row-level labels + existing OOF prediction arrays.
# CPU-only, no GPU, no volume cache -- self-contained, does not assume any
# earlier cell/notebook ran in this kernel session.
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

import config
import evaluate
import model

labels_df = pd.read_csv(config.TRAIN_LABELS_PATH)
family_df = pd.read_csv(config.DATA_PROCESSED / "baseline_features.csv")[
    [config.UID_COLUMN, "inplane_family"]
]
labeled_df = labels_df.merge(family_df, on=config.UID_COLUMN, how="inner").reset_index(drop=True)
y_true = labeled_df[config.TARGET_COLUMN].to_numpy()
family = labeled_df["inplane_family"].to_numpy()

repeat_seeds = list(range(config.SEED, config.SEED + 5))

# NEW production composition (notebooks/26, 6th Opus review): a single
# per-subject-centered CNN variant replaces the old 6-variant ensemble
# entirely.
NEW_VARIANT_PREFIXES = model.PRODUCTION_VARIANT_PREFIXES
assert NEW_VARIANT_PREFIXES == ["coveragefix_persubject"], (
    "model.PRODUCTION_VARIANT_PREFIXES changed since this notebook was written "
    "-- re-check which composition this refit is actually for."
)
new_cnn_arrays = [
    np.load(config.DATA_PROCESSED / f"{prefix}_oof_seed{s}.npy")
    for prefix in NEW_VARIANT_PREFIXES for s in repeat_seeds
]

# OLD shipped composition (6 variants x 5 seeds) -- reference arm for the gate.
OLD_VARIANT_PREFIXES = ["rung3", "rung4_familybias", "rung4_lrsched", "rung4_augment",
                        "rung4_classweight", "rung4_fixedepoch"]
old_cnn_arrays = [
    np.load(config.DATA_PROCESSED / f"{prefix}_oof_seed{s}.npy")
    for prefix in OLD_VARIANT_PREFIXES for s in repeat_seeds
]

baseline_oof_repeats = [np.load(config.DATA_PROCESSED / f"baseline_oof_seed{s}.npy") for s in repeat_seeds]
baseline_pooled = np.mean(baseline_oof_repeats, axis=0)

print(f"{len(new_cnn_arrays)} new-composition CNN members (1 variant x 5 seeds), "
      f"{len(old_cnn_arrays)} old-composition members (6 variants x 5 seeds), "
      f"{len(baseline_oof_repeats)} baseline repeats pooled, {len(labeled_df)} rows.")


5 new-composition CNN members (1 variant x 5 seeds), 30 old-composition members (6 variants x 5 seeds), 5 baseline repeats pooled, 1362 rows.


In [2]:
# [RUN ME] (no new data access -- defines helpers used by every cell below).
# Identical to notebooks/22's op03helpers -- same method, same fold design.
EPS = 1e-6


def to_logit(p):
    p = np.clip(p, EPS, 1 - EPS)
    return np.log(p / (1 - p))


def logit_mean(arrays):
    """Logit-space pooling -- notebook 22's winning pre-registered comparison
    (delta -0.0020 vs. probability-space); not re-tested here since that's
    a property of pooling correlated logits, not of which composition is
    being pooled."""
    return 1.0 / (1.0 + np.exp(-np.mean([to_logit(a) for a in arrays], axis=0)))


FOLDS = evaluate.make_folds(y_true, family, n_splits=config.N_FOLDS, random_state=config.RANDOM_STATE)


def new_unregularized_logreg():
    return LogisticRegression(C=np.inf, max_iter=1000)


def row_cv_score(X, y=y_true, folds=FOLDS):
    """Honest row-wise CV log loss of a logistic-regression calibrated
    blend fit on feature matrix X -- a row's calibration params are never
    fit on data that includes that row."""
    scores = []
    for train_idx, test_idx in folds:
        clf = new_unregularized_logreg()
        clf.fit(X[train_idx], y[train_idx])
        p_test = clf.predict_proba(X[test_idx])[:, 1]
        scores.append(evaluate.log_loss_score(y[test_idx], p_test))
    return np.array(scores)


def full_fit(X, y=y_true):
    """Full-data fit -- the params to actually ship. row_cv_score above is
    the honest number to trust for how well params like these generalize."""
    clf = new_unregularized_logreg()
    clf.fit(X, y)
    return clf.coef_[0], clf.intercept_[0]


def blend_features(cnn_p, baseline_p):
    return np.column_stack([to_logit(cnn_p), to_logit(baseline_p)])


print(f"{len(FOLDS)} row-wise folds built (evaluate.make_folds, stratified on target x family).")


5 row-wise folds built (evaluate.make_folds, stratified on target x family).


In [3]:
# [RUN ME] (no new data access -- uses the arrays built above).
# Refit + formal gate: new (1-variant, per-subject-centered) composition
# vs. the shipped 6-variant recipe, on the identical fold split. Uses BOTH
# notebook 22's row_cv_score/full_fit (honest CV estimate + ship-ready
# constants) AND evaluate.paired_gate (this project's standard row-level
# paired-bootstrap decision rule, min_effect=0.003).
new_cnn_pooled = logit_mean(new_cnn_arrays)
old_cnn_pooled = logit_mean(old_cnn_arrays)

new_scores = row_cv_score(blend_features(new_cnn_pooled, baseline_pooled))
old_scores = row_cv_score(blend_features(old_cnn_pooled, baseline_pooled))
print(f"NEW composition (1 variant, per-subject-centered), row-wise CV: "
      f"mean={new_scores.mean():.4f} sd={new_scores.std(ddof=1):.4f}")
print(f"OLD composition (6 variants, shipped recipe),        row-wise CV: "
      f"mean={old_scores.mean():.4f} sd={old_scores.std(ddof=1):.4f}")
print("for comparison -- notebook 22's originally reported OLD-composition CV: 0.3617")

new_oof = evaluate.oof_predict(blend_features(new_cnn_pooled, baseline_pooled), y_true, FOLDS)
old_oof = evaluate.oof_predict(blend_features(old_cnn_pooled, baseline_pooled), y_true, FOLDS)
gate = evaluate.paired_gate(
    "new per-subject-centered composition vs. shipped 6-variant recipe",
    new_oof, old_oof, y_true, min_effect=0.003,
)

(a, b), c = full_fit(blend_features(new_cnn_pooled, baseline_pooled))
print(f"\nfull-data fit (MAIN BLEND params to ship): a={a:.4f}  b={b:.4f}  c={c:.4f}")


NEW composition (1 variant, per-subject-centered), row-wise CV: mean=0.3076 sd=0.0197
OLD composition (6 variants, shipped recipe),        row-wise CV: mean=0.3617 sd=0.0249
for comparison -- notebook 22's originally reported OLD-composition CV: 0.3617
new per-subject-centered composition vs. shipped 6-variant recipe
  candidate row-CV log loss = 0.3076   reference = 0.3617
  delta (candidate - reference) = -0.0541   95% paired-bootstrap CI = [-0.0762, -0.0322]
  -> CLEARS the gate (adopt) (rule: whole CI < 0 AND |delta| > 0.003)


full-data fit (MAIN BLEND params to ship): a=0.8694  b=0.2765  c=-0.1447


In [4]:
# [RUN ME] (no new data access -- uses the arrays built above).
# CNN-only fallback calibration for submission.combine_predictions's
# degenerate-baseline-mask path -- same method as notebook 22's op07fallback,
# refit on the new composition.
X_cnn_only = to_logit(new_cnn_pooled).reshape(-1, 1)
fallback_cv_scores = row_cv_score(X_cnn_only)
(a1,), c1 = full_fit(X_cnn_only)
raw_cnn_score = evaluate.log_loss_score(y_true, new_cnn_pooled)
raw_cnn_auc = evaluate.auroc_score(y_true, new_cnn_pooled)

print(f"CNN-only fallback calibration (FALLBACK params to ship): a1={a1:.4f}  c1={c1:.4f}")
print(f"row-wise CV log loss (calibrated, CNN-only): mean={fallback_cv_scores.mean():.4f} "
      f"sd={fallback_cv_scores.std(ddof=1):.4f}")
print(f"raw uncalibrated CNN pooled: log loss={raw_cnn_score:.4f}  AUROC={raw_cnn_auc:.4f}")


CNN-only fallback calibration (FALLBACK params to ship): a1=0.9467  c1=-0.1014
row-wise CV log loss (calibrated, CNN-only): mean=0.3131 sd=0.0237
raw uncalibrated CNN pooled: log loss=0.3133  AUROC=0.9381


In [5]:
# [RUN ME] -- the one check that de-risks shipping "auto" centering at
# inference (6th Opus review's top recommendation): proves
# features.striatum_center_mm, called on a freshly-resampled volume exactly
# the way data.load_volume(uid, center_mm="auto") calls it, reproduces
# notebooks/25's op02audit measurements EXACTLY for a sample of training
# volumes -- the 25 shipped checkpoints were trained on those measurements.
# If this is not an exact match, STOP: do not ship "auto" centering until
# the divergence is found and fixed. Real per-uid volume loading (CPU,
# ~2-3 min for 100 volumes) -- prints only an aggregate max-diff and a
# count, never a uid or per-row value, per this project's AI-assistant
# data rule.
import nibabel as nib

import data
import features

checkpoint_path = config.DATA_PROCESSED / "nb25_op02audit_checkpoint.npz"
ckpt = np.load(checkpoint_path, allow_pickle=True)
ckpt_uids = list(ckpt["uids"])
ckpt_offsets = ckpt["offsets_mm"]
assert int(ckpt["next_i"]) == len(ckpt_uids) and int(ckpt["n_degenerate"]) == 0, (
    "nb25_op02audit_checkpoint.npz is incomplete or has degenerate volumes -- "
    "re-run notebooks/25's op02audit cell to completion first."
)

N_SAMPLE = 100
rng = np.random.RandomState(config.RANDOM_STATE)
sample_idx = rng.choice(len(ckpt_uids), size=N_SAMPLE, replace=False)

max_abs_diff = 0.0
n_checked = 0
for i in sample_idx:
    uid = ckpt_uids[i]
    path = config.NIFTI_DIR / f"{uid}.nii.gz"
    img = nib.load(str(path))
    resampled, _ = data.resample_to_spacing(img.get_fdata(), img.affine, config.TARGET_SPACING)
    recomputed = features.striatum_center_mm(resampled, config.TARGET_SPACING)
    assert recomputed is not None, (
        "mask degenerate for a training uid that op02audit found non-degenerate -- "
        "STOP, this should be impossible (same function, same inputs)."
    )
    max_abs_diff = max(max_abs_diff, float(np.max(np.abs(np.asarray(recomputed) - ckpt_offsets[i]))))
    n_checked += 1

print(f"checked {n_checked} random training uids: max|recomputed - nb25 offset| = {max_abs_diff:.10f} mm")
if max_abs_diff < 1e-6:
    print("PASS -- exact match. Safe to ship data.load_volume(uid, center_mm=\"auto\").")
else:
    print("*** STOP: mismatch -- inference-time centroid computation diverges from "
          "training. Do NOT proceed to packaging until this is root-caused. ***")


checked 100 random training uids: max|recomputed - nb25 offset| = 0.0000000000 mm
PASS -- exact match. Safe to ship data.load_volume(uid, center_mm="auto").


**What we're looking for:** (1) does the new per-subject-centered CNN's
blend beat the shipped 6-variant recipe under this project's formal gate,
and what are the full-data-fit constants to hardcode into
`submission_src/main.py`; (2) does recomputing each training volume's
striatum center via the exact production code path
(`features.striatum_center_mm` on a freshly-resampled volume) exactly
reproduce `notebooks/25`'s measurements -- the identity the entire
per-subject-centering approach depends on.

**What we found (run 2026-09-13):**

```
NEW composition (1 variant, per-subject-centered), row-wise CV: mean=0.3076 sd=0.0197
OLD composition (6 variants, shipped recipe),        row-wise CV: mean=0.3617 sd=0.0249

new per-subject-centered composition vs. shipped 6-variant recipe
  candidate row-CV log loss = 0.3076   reference = 0.3617
  delta (candidate - reference) = -0.0541   95% paired-bootstrap CI = [-0.0762, -0.0322]
  -> CLEARS the gate (adopt)

full-data fit (MAIN BLEND params to ship): a=0.8694  b=0.2765  c=-0.1447

CNN-only fallback calibration (FALLBACK params to ship): a1=0.9467  c1=-0.1014
row-wise CV log loss (calibrated, CNN-only): mean=0.3131 sd=0.0237
raw uncalibrated CNN pooled: log loss=0.3133  AUROC=0.9381

op06verify: checked 100 random training uids, max|recomputed - nb25 offset| = 0.0000000000 mm
-> PASS, exact match.
```

The gate clears decisively (delta ~5.4x this project's own `min_effect=0.003`
bar, CI nowhere near 0). The exact-match verification is a clean PASS --
`features.striatum_center_mm`, called via the real production code path
(`data.load_volume(uid, center_mm="auto")`), reproduces `notebooks/25`'s
op02audit measurements bit-for-bit for every one of the 100 sampled
training uids. The fitted constants match the 6th Opus review's own
independent cross-check reconstruction to 4 decimal places exactly (a=0.8694,
b=0.2765, c=-0.1447, a1=0.9467, c1=-0.1014) -- two independent computations
of the identical fit, full agreement.

**Decision:** adopt. `submission_src/main.py`'s BLEND_A/B/C and
FALLBACK_A1/C1 already updated to these values (no change needed --
they matched the provisional cross-check numbers exactly). `data.py`'s
`center_mm="auto"` path is confirmed safe to ship.

**Next step:** rebuild submission assets
(`python scripts/build_submission_assets.py` -- syncs `submission_src/`'s
copies of config.py/data.py/model.py/features.py/dataset.py/submission.py
from `src/`, copies the 25 `coveragefix_persubject_*` checkpoints, refits
the ComBat baseline on 100% of the data), re-sync the separate
runtime-repo clone, local Docker smoke test (`MSYS2_ARG_CONV_EXCL="*"`
workaround from the first submission pass still applies), platform smoke
test (free, watch for 0/N degenerate-centroid fallback triggers), then
the one real submission.